<div align="right" style="text-align: right"><i>Claude Fable 5.2<br>with Peter Norvig<br>Nov 2026</i></div>


<br>
<fieldset>
<p><mark>On Nov. 16, 2026, my Claude account told me that I had some free Fable 5.2 tokens that expired in a day. So I asked  "<i><b>Starting with the notebook <a href="Portmantout.ipynb">Portmantout.ipynb</a> see if you can find a shorter portmantout. Produce a similar notebook, but use a better algorithm, perhaps with an exact solver like some form of linear programming.<b></i>. 

<p><mark>Both Tom Murphy and I formulated the portmantout problem as a variant of the Travelling Salesperson problem, which is NP-hard, and thus we used heuristics that give a good but not optimal solution. Claude formulated it as a min-cost circulation problem (which is polynomial), followed by a conversion from a circulation to a walk. In general there is no guarantee that the circulation can be efficiently converted into an optimal walk, but it just happens that with the given word list it all works out, and thus a provably optimal solution can be computed in just four minutes. 
    
<p><mark>This notebook is the result of the prompt. It is written almost entirely by <a href="https://claude.ai">Claude</a>. I added this preamble, made a couple of one-or-two-word edits for awkward sentences, and added a couple of links to wikipedia articles, but the rest is all Claude. –<i>Peter Norvig</i></mark></p>
</fieldset>

# An Optimal Portmantout

[Another notebook](Portmantout.ipynb) defines a [**portmantout**](http://tom7.org/portmantout/) as a string *S* made by overlapping words from a word list *W*, one after another, such that every word in *W* appears in *S* at least once. Each word must overlap, sharing at least one letter with the word before it. The goal is to find a portmantout of minimal length. For [Tom Murphy's](https://tom7.org/) [list of 108,709 words](wordlist.asc) (931,823 letters) the following programs have found portmantouts with the following number of letters:

|Letters|Author|Method|% of optimal|
|--:|--|:--|--:|
|630,408|Tom Murphy|Greedily joining particles ([portmantout.cc](https://sourceforge.net/p/tom7misc/svn/HEAD/tree/trunk/portmantout/))|117%|
|554,113|Peter Norvig|Greedy path with bridges ([Portmantout.ipynb](https://github.com/norvig/pytudes/blob/main/ipynb/Portmantout.ipynb))|103%|
|543,607|Peter Norvig|The same, with near-subword pseudo-words|101%|
|**536,186**|Claude Fable|Min-circulation and connection|**100%**|

The earlier attempts treated this as a variant of the Traveling Salesperson Problem: visit every word, paying for the letters between them. That's an NP-hard problem, so they settled for greedy heuristics. In this notebook the model is not a tour that visits *nodes* but a tour that traverses *edges* (an [Euler tour](https://en.wikipedia.org/wiki/Eulerian_path)). We can formulate this as the [Directed Rural Postman Problem](https://en.wikipedia.org/wiki/Chinese_postman_problem),
“Find the cheapest walk that crosses every required arc at least once, possibly using optional arcs”, which is NP-hard (Lenstra and Rinnooy Kan, 1976). The required arcs here are the nonsubwords, and the optional arcs are all the other words. The linear program solves a relaxation that drops the requirement that the walk be connected. Its answer is always a lower bound on the true optimum. The NP-hard part all lands in the connecting step. There are two possibilities:

- Zero added excess: the connected flow is a true optimum. That is what happened here: the leftover piece "okeydoke" was linked in to the main piece at zero cost, so 536,186 is proven optimal.
- Positive added excess: the connected flow is a heuristic. The result is then a portmantout within that excess of optimal, but not proven optimal.



In [1]:
from collections import Counter, defaultdict, namedtuple
import heapq, itertools, os, urllib.request
import numpy as np
from scipy.sparse import coo_matrix
from scipy.optimize import linprog

# Words, Subwords, and Nonsubwords

If `'bulleting'` is in *S*, then so are its **subwords** `'bullet'` and `'bulletin'`. So the words we must place are the **nonsubwords**: words that are not inside any other word. 

In [2]:
if not os.path.exists('wordlist.asc'):
    urllib.request.urlretrieve('https://norvig.com/ngrams/wordlist.asc', 'wordlist.asc')

W = set(open('wordlist.asc').read().split())

def subwords(word) -> set:
    """All proper substrings of word."""
    return {word[i:j] for i in range(len(word)) for j in range(i + 1, len(word) + 1)} - {word}

nonsubwords = W - {sub for w in W for sub in subwords(w) & W}

print(f'{len(W):,d} words; {len(nonsubwords):,d} nonsubwords with {sum(map(len, nonsubwords)):,d} letters')

108,709 words; 64,389 nonsubwords with 595,805 letters


# Overlaps as Nodes, Words as Gadgets

Look at a portmantout as an alternating sequence of words and the **overlaps** between them:

          dish  ─sh─  shower  ─er─  erase  ─e─  ear  ─ear─  earth
    S  =  dish        ower          ase         ar          th

Each word adds to *S* its letters minus the overlap in front of it. Now draw a graph in which the *overlaps* are the nodes: `sh`, `er`, `e`, `ear`, and so on. These are the strings that are both a proper prefix of some word and a proper suffix of some word; there are 11,293 of them. 

A word joins these nodes through a little **gadget**: two nodes of its own, `in` and `out`, and three kinds of arc. Here is the gadget for `shower`, whose proper prefixes `s`, `sh`, and `show` and proper suffixes `wer`, `er`, and `r` are overlap nodes (`sho` is not, because no word ends in `sho`):

       s    ──[cost=-1]──┐                                                        ┌──[cost=0]──▶ wer
       sh   ──[cost=-2]──┼──▶ (in shower) ──[cost=+6, flow ≥ 1]──▶ (out shower) ──┼──[cost=0]──▶ er
       show ──[cost=-4]──┘                                                        └──[cost=0]──▶ r

       
- An **entry arc** from each overlap node *p* that is a prefix of the word, with cost `-len(p)`.
- One **middle arc** from `in` to `out`, with cost `len(word)`. If the word is a nonsubword, this arc has a constraint that it must carry at least one unit of flow (subwords have no constraint).
- An **exit arc** to each overlap node that is a suffix of the word, with cost 0.

Any route in through a prefix, across the middle arc, and out through a suffix is one way to use the word: `sh` → `er` is the use in `dish`–`shower`–`erase` above, and `show` → `wer` would be another. Entering through *p* costs `len(word) - len(p)`, exactly the letters the word adds to *S*. Leaving is free, because the next word pays for its own overlap. `shower` has 3 × 3 = 9 such routes but only 3 + 1 + 3 = 7 arcs. More important, the requirement to use every nonsubword is simple: it is a lower bound on the flow through a single arc, the middle one, no matter which of the routes is taken.

A portmantout is then a **walk** in this graph, and its length is the total cost of the walk. So the problem is: **find the cheapest walk that crosses the middle arc of every nonsubword at least once.** The walk may use a word more than once, and may use subwords. 

A walk needs a beginning and an end. So add a `START` node with a free arc to the `in` node of every nonsubword (the first word has no overlap in front of it), an `END` node with a free arc from the `out` node of every nonsubword, and an arc from `END` back to `START` that carries exactly one unit. Now a portmantout is a closed walk through `START`. Counting how many times the walk uses each arc gives a [**circulation**](https://en.wikipedia.org/wiki/Circulation_problem): an amount of flow on every arc such that, at every node, flow in equals flow out.

In [3]:
Arc = namedtuple('Arc', 'tail, head, cost, low, high') # Flow on arc must be between low and high

START, END = '', '$'

def portmantout_arcs(W, nonsubwords) -> list:
    """The arcs of the network: nodes are overlaps (strings), and ('in', w) and ('out', w) for each word."""
    overlaps = ({w[:i] for w in W for i in range(1, len(w))} &
                {w[i:] for w in W for i in range(1, len(w))})
    arcs = [Arc(END, START, 0, 1, 1)]
    for w in sorted(W):
        IN, OUT, required = ('in', w), ('out', w), int(w in nonsubwords)
        arcs.append(Arc(IN, OUT, len(w), required, None))
        arcs.extend(Arc(w[:i], IN, -i, 0, None) for i in range(1, len(w)) if w[:i] in overlaps)
        arcs.extend(Arc(OUT, w[i:], 0, 0, None) for i in range(1, len(w)) if w[i:] in overlaps)
        if required:
            arcs.extend([Arc(START, IN, 0, 0, None), Arc(OUT, END, 0, 0, None)])
    return arcs

arcs = portmantout_arcs(W, nonsubwords)

print(f'{len(arcs):,d} arcs; {len({a.tail for a in arcs}):,d} nodes')

904,142 arcs; 228,711 nodes


# The Cheapest Circulation

Finding the min-cost circulation is a linear program: minimize the total cost of the flow, subject to flow-in equals flow-out at every node, and each flow within its bounds. Linear programs can have fractional answers, but not this one: the constraint matrix of a network flow problem is *totally unimodular*, which guarantees that the simplex method lands on a solution in whole numbers. It takes a few minutes.

The solver also returns a **potential** for each node (the dual variables of the linear program). Think of it as an altitude, chosen so that no arc is a bargain: an arc's cost is never less than the drop in potential from its tail to its head, and is exactly equal to it on every arc where the solution chose to send flow. We'll need the potentials in the next step.

In [4]:
def min_cost_circulation(arcs) -> tuple:
    "The integer flow on each arc that minimizes total cost, and a dict of {node: potential}."
    ids  = {node: i for i, node in enumerate(dict.fromkeys(node for a in arcs for node in (a.tail, a.head)))}
    cols = np.arange(len(arcs))
    A = coo_matrix((np.r_[np.ones(len(arcs)), -np.ones(len(arcs))],
                    (np.r_[[ids[a.tail] for a in arcs], [ids[a.head] for a in arcs]], np.r_[cols, cols])))
    result = linprog([a.cost for a in arcs], A_eq=A.tocsr(), b_eq=np.zeros(len(ids)), 
                     bounds=[(a.low, a.high) for a in arcs], method='highs-ds')
    assert result.status == 0 and np.allclose(result.x, np.rint(result.x))
    return [round(f) for f in result.x], {node: round(result.eqlin.marginals[i]) for node, i in ids.items()}

def total_cost(arcs, flows) -> int: return sum(a.cost * f for a, f in zip(arcs, flows))

%time flows, potential = min_cost_circulation(arcs)

optimum = total_cost(arcs, flows)
print(f'Minimum cost: {optimum:,d} letters')

CPU times: user 3min 39s, sys: 1.62 s, total: 3min 40s
Wall time: 3min 40s
Minimum cost: 536,186 letters


# Connecting the Flow

The flow says *how many times* to use each arc, but not in what order. We will see that the order is easy, as long as the arcs with flow are all **connected** to each other. The linear program does not promise that. Let's see what we got:

In [5]:
def components(arcs, flows) -> list:
    "The sets of nodes that are connected by arcs with nonzero flow, largest first."
    neighbors = defaultdict(list)
    for a, f in zip(arcs, flows):
        if f:
            neighbors[a.tail].append(a.head)
            neighbors[a.head].append(a.tail)
    result, seen = [], set()
    for node in neighbors:
        if node not in seen:
            component, frontier = {node}, [node]
            while frontier:
                for n in neighbors[frontier.pop()]:
                    if n not in component:
                        component.add(n)
                        frontier.append(n)
            seen |= component
            result.append(component)
    return sorted(result, key=len, reverse=True)

[(len(c), list(c)[:3]) for c in components(arcs, flows)]

[(134803, ['', ('out', 'baizes'), ('in', 'hawkweeds')]),
 (3, [('out', 'okeydoke'), ('in', 'okeydoke'), 'oke'])]

Almost: there is one big component, and one stray. The word `okeydoke` overlaps *itself*, on `oke`, and nothing else uses that overlap, so the cheapest thing for the linear program to do is to have `okeydoke` chase its own tail forever, saving three letters without ever joining the rest of the portmantout.

We need to reroute some flow to link the stray to the main component. The potentials tell us what that costs. Define the **excess** of an arc as its cost minus its drop in potential; that is never negative. Around any closed walk the drops in potential sum to zero, so if we add a unit of flow all the way around a closed walk, the total cost goes up by exactly the sum of the excesses. We can also *remove* a unit of flow from an arc that has some to spare; all such arcs have zero excess. So the function `connect` uses Dijkstra's algorithm to find the least-excess closed walk that goes from the stray component to the main component and back, and changes the flow along it. If the excess is zero, we have found a different solution to the linear program that is just as good, but connected.

In [6]:
def connect(arcs, flows, potential):
    "Mutate flows to be connected, by changing the flow along the cheapest closed walk out of each stray component."
    forward, backward = defaultdict(list), defaultdict(list) # {node: [(node2, excess, arc_index, change)]}
    for i, a in enumerate(arcs):
        if a.high is None:
            excess = a.cost - potential[a.tail] + potential[a.head]
            forward[a.tail].append((a.head, excess, i, +1));  forward[a.head].append((a.tail, 0, i, -1))
            backward[a.head].append((a.tail, excess, i, +1)); backward[a.tail].append((a.head, 0, i, -1))
    while len(comps := components(arcs, flows)) > 1:
        main, stray = comps[0], comps[-1]
        def walks():
            for c in stray:
                (d1, via1), (d2, via2) = shortest_paths(c, forward, flows), shortest_paths(c, backward, flows)
                for m in main & set(d1) & set(d2):
                    yield d1[m] + d2[m], c, m, via1, via2
        excess, c, m, via1, via2 = min(walks(), key=lambda walk: walk[0])
        for via in (via1, via2): # Trace the paths from c to m and from m to c
            node = m
            while node != c:
                i, change = via[node]
                flows[i] += change
                node = arcs[i].tail if (via is via1) == (change == +1) else arcs[i].head
        print(f'Linked {len(stray)} nodes to the main component; excess {excess}')

def shortest_paths(source, graph, flows) -> tuple:
    """Dijkstra's algorithm: for nodes reachable from source: 
    {node: distance}, {node: (arc_index, change) used to get there}."""
    dist, via, frontier, ticket = {source: 0}, {}, [(0, 0, source)], itertools.count(1)
    while frontier:
        d, _, node = heapq.heappop(frontier)
        if d == dist[node]:
            for (node2, excess, i, change) in graph[node]:
                if d + excess < dist.get(node2, float('inf')) and (change == +1 or flows[i] > arcs[i].low):
                    dist[node2], via[node2] = d + excess, (i, change)
                    heapq.heappush(frontier, (d + excess, next(ticket), node2))
    return dist, via

connect(arcs, flows, potential)

assert total_cost(arcs, flows) == optimum

Linked 3 nodes to the main component; excess 0


The stray is linked in at no cost. (It turns out that `okeydoke` now follows `chokey`, overlapping by `okey`, which saves just as many letters as before.)

# From Circulation to Walk: Euler's Theorem

Now for the order. Make a multigraph with *f* copies of each arc that carries flow *f*. Every node has as many arcs in as out, and the multigraph is connected. Euler showed in 1736 that such a graph has an **Euler circuit**: a closed walk that uses every arc exactly once. [Hierholzer's algorithm](https://en.wikipedia.org/wiki/Eulerian_path#Hierholzer's_algorithm) finds one in linear time: follow unused arcs until stuck (which can only happen back at the start), then back up, splicing in a detour wherever a node on the walk still has unused arcs. With the approach used in this notebook, any order in which the arcs can be walked is as good as any other.

In [7]:
def euler_circuit(arcs, flows, start=START) -> list:
    "A list of nodes, starting and ending at start, that uses each arc `flow` times."
    successors = defaultdict(list)
    for a, f in zip(arcs, flows):
        successors[a.tail].extend([a.head] * f)
    stack, circuit = [start], []
    while stack:
        if successors[stack[-1]]:
            stack.append(successors[stack[-1]].pop())
        else:
            circuit.append(stack.pop())
    assert len(circuit) == sum(flows) + 1
    return circuit[::-1]

circuit = euler_circuit(arcs, flows)

circuit[:11], circuit[-5:] # The start and end of the circuit

(['',
  ('in', 'jealousies'),
  ('out', 'jealousies'),
  's',
  ('in', 'syzygies'),
  ('out', 'syzygies'),
  's',
  ('in', 'systoles'),
  ('out', 'systoles'),
  'les',
  ('in', 'lest')],
 ['im', ('in', 'immutably'), ('out', 'immutably'), '$', ''])

The circuit is a portmantout. It visits nodes in the order *overlap*, `('in', word)`, `('out', word)`, *overlap*, ..., so each `'in'` node, together with the overlap before it, is one **step**:

In [8]:
def steps(circuit) -> list:
    """The (overlap, word) steps of the circuit."""
    return [(overlap, node[1]) for overlap, node in zip(circuit, circuit[1:]) 
            if isinstance(node, tuple) and node[0] == 'in']

def portman(P) -> str: 
    """The string formed by a list of (overlap, word) steps."""
    return ''.join(word[len(overlap):] for overlap, word in P)

P = steps(circuit)
S = portman(P)

print(f'S has {len(S):,d} letters; P has {len(P):,d} steps')

S has 536,186 letters; P has 99,671 steps


# Verifying

Let's check everything that could go wrong: every step is a word; each overlap is a proper nonempty prefix of its word and a proper suffix of the previous word; and every word in *W* is in *S*.

In [9]:
def is_portmantout(P, W) -> bool:
    """Is P a legal path of (overlap, word) steps whose string contains every word in W?"""
    S = portman(P)
    return (P[0][0] == '' 
            and all(word in W for _, word in P)
            and all(0 < len(o) < min(len(prev), len(word)) and prev.endswith(o) and word.startswith(o)
                    for (_, prev), (o, word) in zip(P, P[1:]))
            and all(w in S for w in W))

assert is_portmantout(P, W)
assert len(S) == optimum

open('portmantout_optimal.txt', 'w').write(S)

print(S[:300] + ' ... ' + S[-300:])

jealousiesyzygiesystolestzurisusurrusesystemizedwindlingyrustlinglypticsystematizingyrosewoodsystematizesystematizedwindlestzitzisystematizationovocainextricablyurtsyrupyxisthmusesyringesyrensyphonsyphonedwindledwightsyphiloidolsyphilizedweltsyntaxesynoviaswooninglyphsynoviallingyroscopicallysinsyno ... eedfulnessabotagescaladedabblestacetonicacciatoreadorsabotagedabblersabinesabersaberingabalonesomelycanthroperiesaberedabbledabbedamnederailedabbingabaftercarefreezingabacusesaberrancyanicacaoscarsaberdeenabbingabacillaryngalahadsabbatsabbaticalsabbathsabbatialabamalgamatesacapulcoachmanaheimmutably


# Why Nothing Shorter Is Possible

The claim is that **no portmantout of *W* has fewer than 536,186 letters.** The argument has three parts:

1. **Every portmantout is a walk that steps on every nonsubword.** Suppose a nonsubword *w* appears in *S* without being one of the steps. As we move along the steps, their start positions and their end positions both strictly increase. So *w* fits between two consecutive steps: one that starts before *w* does and ends inside it, and one that starts inside *w* and ends after it. (Any steps that lie entirely inside *w* are subwords of *w*, and can be dropped.) That means *w* overlaps both neighbors, and we can insert it as a step without changing *S*.
2. **Every such walk is a circulation with the same cost.** One unit of flow from `START` to the first word; one unit through each word and each overlap, in order; one from the last word to `END` and back to `START`. The cost is exactly the number of letters. (If the walk starts or ends with a subword, dropping it gives a shorter portmantout, so `START` and `END` only need arcs to nonsubwords.) 
3. **The linear program finds the cheapest circulation,** and our *S* has exactly that many letters. (Linking in `okeydoke` cost nothing.)

So the 536,186 is not just the best this method can do; it is the best *any* method can do. (There are many different portmantouts of that length; a different Euler circuit of the same flow gives another one.)

# What an Optimal Portmantout Looks Like

How often is each word used? And where do the letters go? What is [the whole portmantout](portmantout_optimal.txt)?

In [10]:
uses    = Counter(word for _, word in P)
extra   = Counter({w: n - (w in nonsubwords) for w, n in uses.items() if n > (w in nonsubwords)})
bridge_letters = sum(n * len(w) for w, n in extra.items())

print(f'{len(nonsubwords):,d} required steps + {sum(extra.values()):,d} bridge steps = {len(P):,d} steps')
print(f'{len(extra):,d} distinct bridge words; {sum(1 for w in extra if w in nonsubwords)} of them are nonsubwords used again')
print(f'Most common bridges: {dict(extra.most_common(20))}')

64,389 required steps + 35,282 bridge steps = 99,671 steps
527 distinct bridge words; 34 of them are nonsubwords used again
Most common bridges: {'sic': 3029, 'sub': 2919, 'of': 2290, 'do': 1835, 'lyre': 1820, 'so': 1756, 'gum': 1665, 'yep': 1456, 'sin': 1353, 'gun': 1242, 'rye': 861, 'sun': 831, 'sow': 638, 'lest': 494, 'type': 459, 'spa': 415, 'typo': 414, 'she': 349, 'yow': 281, 'tv': 246}


In [11]:
overlaps = Counter(len(o) for o, _ in P[1:])
total    = sum(n * k for k, n in overlaps.items())

print(f'{sum(map(len, nonsubwords)):,d} letters in nonsubwords + {bridge_letters:,d} in bridges '
      f'- {total:,d} overlapped = {len(S):,d}')
print(f'Average overlap: {total / (len(P) - 1):.2f} letters')
for k in sorted(overlaps):
    print(f'{k:2d}-letter overlaps: {overlaps[k]:6,d}')

595,805 letters in nonsubwords + 117,065 in bridges - 176,684 overlapped = 536,186
Average overlap: 1.77 letters
 1-letter overlaps: 61,403
 2-letter overlaps: 19,073
 3-letter overlaps: 10,295
 4-letter overlaps:  4,500
 5-letter overlaps:  1,937
 6-letter overlaps:    944
 7-letter overlaps:    573
 8-letter overlaps:    385
 9-letter overlaps:    182
10-letter overlaps:    163
11-letter overlaps:    102
12-letter overlaps:     67
13-letter overlaps:     30
14-letter overlaps:     14
15-letter overlaps:      2


The bridges are almost all tiny words that turn one letter into another: 45% of the nonsubwords end in `s`, far more than start with `s`, so the optimal solution spends thousands of two- and three-letter words, like `sic`, `sub`, and `so`, converting a final `s` into something more useful. 